# A4: Visualisation — Criticality & Vulnerability

Run `criticality_vulnerability.ipynb` first to generate `bridge_analysis.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

df = pd.read_csv('../data/data_processed/bridge_analysis.csv')
df = df[df['condition_weight'].notna() & df['criticality_score'].notna()]
print(f'Bridges loaded: {len(df)}')
df.head(3)

## Plot 1: Criticality vs Vulnerability scatter (per road)

In [ ]:
roads = df['road'].unique()
colours = plt.cm.tab10(np.linspace(0, 1, len(roads)))
road_colour = dict(zip(roads, colours))

fig, ax = plt.subplots(figsize=(10, 7))

for road in roads:
    sub = df[df['road'] == road]
    ax.scatter(sub['criticality_score'], sub['vulnerability_score'],
               label=road, color=road_colour[road], alpha=0.7, s=40)

# Label top-5 most critical
top5 = df.nlargest(5, 'criticality_score')
for _, row in top5.iterrows():
    ax.annotate(row['name'][:20], (row['criticality_score'], row['vulnerability_score']),
                fontsize=7, ha='left', va='bottom')

ax.set_xlabel('Criticality Score (betweenness × daily tonnes)', fontsize=12)
ax.set_ylabel('Vulnerability Score (condition weight)', fontsize=12)
ax.set_title('Bridge Criticality vs Vulnerability', fontsize=14)
ax.legend(title='Road', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../data/data_processed/criticality_vs_vulnerability.png', dpi=150)
plt.show()

## Plot 2: Geographic map — bubble size = criticality, colour = condition

In [ ]:
condition_colours = {'A': 'green', 'B': 'yellow', 'C': 'orange', 'D': 'red'}

# Normalise criticality to bubble size (min 10, max 300)
crit = df['criticality_score'].values
if crit.max() > 0:
    sizes = 10 + 290 * (crit - crit.min()) / (crit.max() - crit.min() + 1e-9)
else:
    sizes = np.full(len(crit), 20)

fig, ax = plt.subplots(figsize=(8, 12))

for cond, colour in condition_colours.items():
    mask = df['condition'] == cond
    ax.scatter(df[mask]['lon'], df[mask]['lat'],
               s=sizes[mask], c=colour, alpha=0.6, label=f'Condition {cond}', edgecolors='grey', linewidths=0.3)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Bridges: size = criticality, colour = condition', fontsize=13)
ax.legend(title='Condition')
plt.tight_layout()
plt.savefig('../data/data_processed/geo_criticality_map.png', dpi=150)
plt.show()

## Plot 3: Top-10 most critical bridges (bar chart)

In [ ]:
top10_crit = df.nlargest(10, 'criticality_score').copy()
top10_crit['label'] = top10_crit['name'].str[:25] + ' (' + top10_crit['road'] + ')'

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10_crit['label'], top10_crit['criticality_score'],
               color=[condition_colours.get(c, 'grey') for c in top10_crit['condition']])
ax.set_xlabel('Criticality Score (betweenness × daily tonnes)')
ax.set_title('Top-10 Most Critical Bridges')
ax.invert_yaxis()

# Legend for condition colours
from matplotlib.patches import Patch
legend_patches = [Patch(color=v, label=f'Condition {k}') for k, v in condition_colours.items()]
ax.legend(handles=legend_patches, loc='lower right')

plt.tight_layout()
plt.savefig('../data/data_processed/top10_critical.png', dpi=150)
plt.show()

## Plot 4: Top-10 most vulnerable bridges (bar chart)

In [ ]:
top10_vuln = df.nlargest(10, 'vulnerability_score').copy()
top10_vuln['label'] = top10_vuln['name'].str[:25] + ' (' + top10_vuln['road'] + ')'

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10_vuln['label'], top10_vuln['vulnerability_score'],
        color=[condition_colours.get(c, 'grey') for c in top10_vuln['condition']])
ax.set_xlabel('Vulnerability Score (condition weight × flood risk)')
ax.set_title('Top-10 Most Vulnerable Bridges')
ax.invert_yaxis()
ax.legend(handles=legend_patches, loc='lower right')
plt.tight_layout()
plt.savefig('../data/data_processed/top10_vulnerable.png', dpi=150)
plt.show()